# Convolutional Neural Networks

Import of the used packages and methods

In [2]:
import numpy as np
import tensorflow as tf
from pyparsing import Dict
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import pandas as pd

I0000 00:00:1787144164.344363  679699 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787144164.344804  679699 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787144164.373762  679699 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787144165.052277  679699 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

# Split Train/Test in 5 folds
splitted to Dicts of:
- meta_tr (train)
- meta_te (test)

and each group will appear exactly once in the test set across all folds

In [22]:
from sklearn.model_selection import GroupKFold
from auswertung_features.build_features import split_into_label_blocks, load_session

meta = pd.read_csv("../auswertung_features/meta.csv")
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
meta_tr: dict[int, pd.DataFrame] = {}
meta_te: dict[int, pd.DataFrame] = {}

dict_blocks = {}
meta_blocks = {"subject_id":[], "hand": [], "block_id":[]}
for i, r in meta.iterrows():
        csv_path = r["rel_path"]
        df_session = load_session(csv_path)

        #Error Handling für leere dfs
        if df_session is None:
            continue

        blocks = split_into_label_blocks(df_session)
        for block in blocks:
            meta_blocks["subject_id"].append(r["subject_id"])
            meta_blocks["hand"].append(r["hand"])
            meta_blocks["block_id"].append(len(dict_blocks))
            dict_blocks[len(dict_blocks)] = block
meta_blocks = pd.DataFrame(meta_blocks)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(meta_blocks, groups=meta_blocks["subject_id"])):
    meta_tr[fold] = meta_blocks.iloc[tr_idx].reset_index(drop=True)
    meta_te[fold] = meta_blocks.iloc[te_idx].reset_index(drop=True)


[WARN] ../ArduinoApps/emg_messung/python/messdaten/messung_l_170.csv: is empty


## Convert dataset to match TensorFlow Conventions

Input needs to be in shape (batch, steps, features). My own blocks (ndarray) are in the shape of (sample_length, sensor_channels). The steps are the sequence length which should be learned.

# Model class, 1d CNN

In [ ]:
from tensorflow.keras import layers, models

inputs = layers.Input(shape=(None, 4)) # input are steps and features which equals to sample_length and sensor_channels

x = layers.Conv1D(32, activation='relu')(inputs)
